# 🖥️ WMI (Windows Management Instrumentation) Kapsamlı Python Rehberi

Bu notebook, Microsoft'un **Windows Management Instrumentation (WMI)** altyapısını Python `wmi` kütüphanesi üzerinden kullanarak **donanım, yazılım, işletim sistemi, ağ, servisler, işlemler (process), disk, bellek ve kullanıcı yönetimi** konularında derinlemesine sorgulama yapmayı öğreten kapsamlı bir rehberdir.

> **⚠️ Not:** Bu rehber yalnızca **Windows** işletim sistemi üzerinde çalışır. WMI, Windows'a özgü bir teknolojidir.

## 1. WMI Nedir? (Derinlemesine)

**WMI (Windows Management Instrumentation)**, Microsoft'un Windows işletim sistemlerinde bulunan, sistem yönetimi ve izleme işlemleri için merkezi bir altyapı sağlayan teknolojisidir. Kökeni, endüstri standardı olan **CIM (Common Information Model)** ve **WBEM (Web-Based Enterprise Management)** spesifikasyonlarına dayanır.

### 1.1. WMI'ın Temel Bileşenleri

| Bileşen | Açıklama |
|---------|----------|
| **WMI Servisi (`winmgmt.exe`)** | WMI altyapısını yöneten Windows servisi. Sorguları alır, ilgili sağlayıcıya (provider) yönlendirir ve sonuçları döndürür. |
| **CIM Repository** | WMI sınıflarının tanımlarını, ilişkilerini ve örnek verilerini saklayan veritabanı. `%systemroot%\System32\wbem\Repository` konumunda bulunur. |
| **WMI Providers (Sağlayıcılar)** | Her bir veri kaynağı için özel DLL'lerdir. Örneğin donanım bilgileri `cimwin32.dll` sağlayıcısından, performans verileri `perfproc.dll`'den gelir. |
| **WQL (WMI Query Language)** | SQL'e benzer sorgu dili. `SELECT`, `FROM`, `WHERE`, `LIKE`, `OR`, `AND` gibi operatörleri destekler. |
| **Namespace (Ad Alanı)** | WMI sınıflarının mantıksal gruplandırması. Varsayılan: `root\cimv2` |

### 1.2. WMI Çalışma Akışı

```
Python Script (wmi kütüphanesi)
         │
         ▼
COM Arayüzü (pywin32) ── IWbemServices arayüzü
         │
         ▼
WMI Servisi (winmgmt.exe)
         │
         ├── WQL sorgusunu ayrıştırır (parse)
         ├── İlgili Provider'ı bulur
         ▼
Provider DLL (örn. cimwin32.dll)
         │
         ├── Windows Kernel API'lerini çağırır
         ├── Registry'den bilgi okur
         ├── Performans sayaçlarını sorgular
         ▼
Sonuçlar COM üzerinden Python'a döner
```

### 1.3. WMI Namespace'leri

| Namespace | Açıklama |
|-----------|----------|
| `root\cimv2` | En çok kullanılan namespace — donanım, işletim sistemi, kullanıcılar, servisler |
| `root\SecurityCenter2` | Antivirüs, güvenlik duvarı ve antispyware bilgileri |
| `root\Microsoft\Windows\Storage` | Gelişmiş disk ve depolama yönetimi |
| `root\StandardCimv2` | Ağ yapılandırması (NetAdapter, NetIPAddress vb.) |

## 2. Python `wmi` Kütüphanesi Kurulumu ve Temelleri

### Kurulum
```bash
pip install wmi
```
Bu komut `wmi` paketinin yanı sıra bağımlılığı olan `pywin32` paketini de otomatik olarak yükler.

### Temel Kullanım Yapısı
```python
import wmi

c = wmi.WMI()                              # WMI bağlantısı kur
sonuclar = c.Win32_SinifAdi()              # WMI sınıfını sorgula
for sonuc in sonuclar:                     # Sonuçları döngüyle işle
    print(sonuc.OzellikAdi)                # Sınıfın özelliklerine eriş
```

### Uzak Bilgisayara Bağlanma
```python
# Ağdaki başka bir Windows bilgisayarına WMI üzerinden bağlanma
c = wmi.WMI(computer="192.168.1.100", user="admin", password="sifre123")
```

## 3. Donanım Bilgileri Sorgulama

### 3.1. İşlemci (CPU) Bilgileri — `Win32_Processor`

In [ ]:
import wmi
c = wmi.WMI()

print("═" * 60)
print("İŞLEMCİ (CPU) BİLGİLERİ")
print("═" * 60)

for cpu in c.Win32_Processor():
    print(f"  Model           : {cpu.Name}")
    print(f"  Üretici         : {cpu.Manufacturer}")
    print(f"  Çekirdek Sayısı : {cpu.NumberOfCores}")
    print(f"  Thread Sayısı   : {cpu.NumberOfLogicalProcessors}")
    print(f"  Maks Hız (MHz)  : {cpu.MaxClockSpeed}")
    print(f"  Anlık Hız (MHz) : {cpu.CurrentClockSpeed}")
    print(f"  Soket           : {cpu.SocketDesignation}")
    print(f"  L2 Cache (KB)   : {cpu.L2CacheSize}")
    print(f"  L3 Cache (KB)   : {cpu.L3CacheSize}")
    print(f"  Mimari          : {'x64' if cpu.AddressWidth == 64 else 'x86'}")
    print(f"  CPU Yükü (%)    : {cpu.LoadPercentage}")

### 3.2. Bellek (RAM) Bilgileri — `Win32_PhysicalMemory`

In [ ]:
import wmi
c = wmi.WMI()

print("═" * 60)
print("BELLEK (RAM) BİLGİLERİ")
print("═" * 60)

toplam_ram = 0
for i, ram in enumerate(c.Win32_PhysicalMemory(), 1):
    boyut_gb = int(ram.Capacity) / (1024**3)
    toplam_ram += boyut_gb
    
    # MemoryType kodunu metne çevirme
    ram_tipleri = {20: "DDR", 21: "DDR2", 24: "DDR3", 26: "DDR4", 34: "DDR5"}
    ram_tipi = ram_tipleri.get(ram.SMBIOSMemoryType, f"Bilinmiyor ({ram.SMBIOSMemoryType})")
    
    print(f"\n  --- Slot #{i} ---")
    print(f"  Üretici    : {ram.Manufacturer}")
    print(f"  Kapasite   : {boyut_gb:.0f} GB")
    print(f"  Hız (MHz)  : {ram.Speed}")
    print(f"  Tür        : {ram_tipi}")
    print(f"  Konum      : {ram.DeviceLocator}")
    print(f"  Parça No   : {ram.PartNumber.strip() if ram.PartNumber else 'Bilinmiyor'}")

print(f"\n  Toplam RAM : {toplam_ram:.0f} GB")

### 3.3. Ekran Kartı (GPU) Bilgileri — `Win32_VideoController`

In [ ]:
import wmi
c = wmi.WMI()

print("═" * 60)
print("EKRAN KARTI (GPU) BİLGİLERİ")
print("═" * 60)

for gpu in c.Win32_VideoController():
    vram_mb = int(gpu.AdapterRAM) / (1024**2) if gpu.AdapterRAM else 0
    print(f"\n  Model            : {gpu.Name}")
    print(f"  Üretici          : {gpu.AdapterCompatibility}")
    print(f"  VRAM             : {vram_mb:.0f} MB")
    print(f"  Sürücü Versiyonu : {gpu.DriverVersion}")
    print(f"  Sürücü Tarihi   : {gpu.DriverDate}")
    print(f"  Çözünürlük      : {gpu.CurrentHorizontalResolution}x{gpu.CurrentVerticalResolution}")
    print(f"  Yenileme Hızı   : {gpu.CurrentRefreshRate} Hz")
    print(f"  Durum           : {gpu.Status}")

### 3.4. Disk Bilgileri — `Win32_DiskDrive` ve `Win32_LogicalDisk`

In [ ]:
import wmi
c = wmi.WMI()

print("═" * 60)
print("FİZİKSEL DİSK BİLGİLERİ")
print("═" * 60)

for disk in c.Win32_DiskDrive():
    boyut_gb = int(disk.Size) / (1024**3) if disk.Size else 0
    print(f"\n  Model      : {disk.Model}")
    print(f"  Kapasite   : {boyut_gb:.1f} GB")
    print(f"  Arayüz     : {disk.InterfaceType}")
    print(f"  Seri No    : {disk.SerialNumber.strip() if disk.SerialNumber else 'Bilinmiyor'}")
    print(f"  Bölüm Sayı : {disk.Partitions}")

print("\n" + "═" * 60)
print("MANTIKSAL DİSK BÖLÜMLERİ (Sürücüler)")
print("═" * 60)

for disk in c.Win32_LogicalDisk():
    if disk.Size:  # Yalnızca boyutu olan (bağlı) sürücüleri göster
        toplam_gb = int(disk.Size) / (1024**3)
        bos_gb = int(disk.FreeSpace) / (1024**3) if disk.FreeSpace else 0
        kullanilan_gb = toplam_gb - bos_gb
        yuzde = (kullanilan_gb / toplam_gb) * 100
        
        # DriveType kodlarını metne çevirme
        surucu_tipleri = {0: "Bilinmiyor", 1: "Kök Yok", 2: "Çıkarılabilir", 
                         3: "Sabit Disk", 4: "Ağ Sürücüsü", 5: "CD/DVD", 6: "RAM Disk"}
        tip = surucu_tipleri.get(disk.DriveType, "Bilinmiyor")
        
        print(f"\n  {disk.Caption} ({tip})")
        print(f"  Dosya Sistemi : {disk.FileSystem}")
        print(f"  Toplam        : {toplam_gb:.1f} GB")
        print(f"  Kullanılan    : {kullanilan_gb:.1f} GB ({yuzde:.1f}%)")
        print(f"  Boş           : {bos_gb:.1f} GB")

### 3.5. Anakart Bilgileri — `Win32_BaseBoard`

In [ ]:
import wmi
c = wmi.WMI()

print("═" * 60)
print("ANAKART BİLGİLERİ")
print("═" * 60)

for board in c.Win32_BaseBoard():
    print(f"  Üretici     : {board.Manufacturer}")
    print(f"  Model       : {board.Product}")
    print(f"  Seri No     : {board.SerialNumber}")
    print(f"  Versiyon    : {board.Version}")

print("\n" + "═" * 60)
print("BIOS BİLGİLERİ")
print("═" * 60)

for bios in c.Win32_BIOS():
    print(f"  Üretici     : {bios.Manufacturer}")
    print(f"  Versiyon    : {bios.SMBIOSBIOSVersion}")
    print(f"  Tarih       : {bios.ReleaseDate}")
    print(f"  Seri No     : {bios.SerialNumber}")

## 4. İşletim Sistemi Bilgileri

### 4.1. İşletim Sistemi Detayları — `Win32_OperatingSystem`

In [ ]:
import wmi
c = wmi.WMI()

print("═" * 60)
print("İŞLETİM SİSTEMİ BİLGİLERİ")
print("═" * 60)

for os_info in c.Win32_OperatingSystem():
    toplam_ram_gb = int(os_info.TotalVisibleMemorySize) / (1024**2)
    bos_ram_gb = int(os_info.FreePhysicalMemory) / (1024**2)
    
    print(f"  İsim               : {os_info.Caption}")
    print(f"  Versiyon           : {os_info.Version}")
    print(f"  Build Numarası     : {os_info.BuildNumber}")
    print(f"  Mimari             : {os_info.OSArchitecture}")
    print(f"  Bilgisayar Adı     : {os_info.CSName}")
    print(f"  Kayıtlı Kullanıcı  : {os_info.RegisteredUser}")
    print(f"  Son Önyükleme      : {os_info.LastBootUpTime}")
    print(f"  Toplam Fiziksel RAM : {toplam_ram_gb:.2f} GB")
    print(f"  Boş Fiziksel RAM   : {bos_ram_gb:.2f} GB")
    print(f"  Windows Dizini     : {os_info.WindowsDirectory}")

## 5. Ağ Bilgileri

### 5.1. Ağ Adaptörleri — `Win32_NetworkAdapterConfiguration`

In [ ]:
import wmi
c = wmi.WMI()

print("═" * 60)
print("AĞ ADAPTÖRÜ BİLGİLERİ (Aktif Olanlar)")
print("═" * 60)

for adapter in c.Win32_NetworkAdapterConfiguration(IPEnabled=True):
    print(f"\n  Açıklama     : {adapter.Description}")
    print(f"  MAC Adresi   : {adapter.MACAddress}")
    print(f"  IP Adresi    : {adapter.IPAddress}")
    print(f"  Alt Ağ Maskesi : {adapter.IPSubnet}")
    print(f"  Varsayılan Ağ Geçidi : {adapter.DefaultIPGateway}")
    print(f"  DNS Sunucuları : {adapter.DNSServerSearchOrder}")
    print(f"  DHCP Aktif mi? : {'Evet' if adapter.DHCPEnabled else 'Hayır'}")
    if adapter.DHCPEnabled:
        print(f"  DHCP Sunucusu : {adapter.DHCPServer}")

## 6. Servisler ve İşlemler (Process)

### 6.1. Windows Servisleri — `Win32_Service`

In [ ]:
import wmi
c = wmi.WMI()

print("═" * 60)
print("ÇALIŞAN SERVİSLER (İlk 10)")
print("═" * 60)

calisan = [s for s in c.Win32_Service() if s.State == "Running"]
print(f"\nToplam çalışan servis sayısı: {len(calisan)}\n")

for servis in calisan[:10]:
    print(f"  {servis.DisplayName}")
    print(f"    Durum       : {servis.State}")
    print(f"    Başlatma    : {servis.StartMode}")
    print(f"    Çalıştıran  : {servis.StartName}")
    print(f"    PID         : {servis.ProcessId}")
    print()

### 6.2. Çalışan İşlemler (Processes) — `Win32_Process`

In [ ]:
import wmi
c = wmi.WMI()

print("═" * 60)
print("EN ÇOK BELLEK KULLANAN İŞLEMLER (İlk 10)")
print("═" * 60)

islemler = []
for proc in c.Win32_Process():
    if proc.WorkingSetSize:
        islemler.append({
            "Ad": proc.Name,
            "PID": proc.ProcessId,
            "Bellek_MB": int(proc.WorkingSetSize) / (1024**2)
        })

# Bellek kullanımına göre sırala
islemler.sort(key=lambda x: x["Bellek_MB"], reverse=True)

print(f"{'Sıra':<5} {'İşlem Adı':<35} {'PID':<8} {'Bellek (MB)':<12}")
print("-" * 60)
for i, p in enumerate(islemler[:10], 1):
    print(f"{i:<5} {p['Ad']:<35} {p['PID']:<8} {p['Bellek_MB']:.1f}")

## 7. Yüklü Yazılımlar ve Sürücüler

### 7.1. Yüklü Programlar — `Win32_Product`

> **⚠️ Dikkat:** `Win32_Product` sorgusu, her çağrıldığında MSI paketlerini tutarlılık açısından doğrular ve bu işlem yavaş olabilir. Hızlı sonuç için `Win32_InstalledWin32Program` veya Registry sorgusu tercih edilebilir.

In [ ]:
import wmi
c = wmi.WMI()

print("═" * 60)
print("YÜKLÜ SÜRÜCÜLER (İlk 10) — Win32_PnPSignedDriver")
print("═" * 60)

# Win32_PnPSignedDriver, PnP cihazlarının sürücü versiyonlarını içerir
drivers = list(c.Win32_PnPSignedDriver())
print(f"\nToplam imzalı sürücü: {len(drivers)}\n")

for drv in drivers[:10]:
    if drv.DeviceName:
        print(f"  Cihaz         : {drv.DeviceName}")
        print(f"  Sürücü Ver.   : {drv.DriverVersion}")
        print(f"  Sürücü Tarihi : {drv.DriverDate}")
        print(f"  Sağlayıcı     : {drv.DriverProviderName}")
        print(f"  İmzalı mı?    : {'Evet' if drv.IsSigned else 'Hayır'}")
        print()

## 8. WQL (WMI Query Language) ile Özel Sorgular

WQL, SQL'e çok benzer bir sorgu dilidir ve `wmi` kütüphanesinden doğrudan kullanılabilir.

In [ ]:
import wmi
c = wmi.WMI()

# ---- 1. Belirli bir cihaz sınıfını sorgulama ----
print("Intel cihazları (Win32_PnPEntity WHERE Manufacturer LIKE '%Intel%'):")
print("=" * 60)

intel_devices = c.query("SELECT Name, Manufacturer FROM Win32_PnPEntity WHERE Manufacturer LIKE '%Intel%'")
for dev in intel_devices[:5]:
    print(f"  {dev.Name} — {dev.Manufacturer}")

# ---- 2. Durdurulan servisleri bulma ----
print("\n\nDurmuş olan önemli servisler:")
print("=" * 60)

durmus = c.query("SELECT DisplayName, State FROM Win32_Service WHERE State = 'Stopped' AND StartMode = 'Auto'")
for s in durmus[:5]:
    print(f"  {s.DisplayName} — Durum: {s.State}")

# ---- 3. Belirli bir işlemi (process) bulma ----
print("\n\nPython işlemleri:")
print("=" * 60)

python_procs = c.query("SELECT Name, ProcessId, WorkingSetSize FROM Win32_Process WHERE Name LIKE '%python%'")
for p in python_procs:
    bellek_mb = int(p.WorkingSetSize) / (1024**2) if p.WorkingSetSize else 0
    print(f"  {p.Name} (PID: {p.ProcessId}) — Bellek: {bellek_mb:.1f} MB")

## 9. WMI Olay İzleme (Event Watching)

WMI'ın en güçlü özelliklerinden biri, sistem olaylarını **gerçek zamanlı** izleyebilmesidir. Örneğin bir USB cihaz takıldığında veya bir işlem başlatıldığında bildirim alabilirsiniz.

### 9.1. İşlem Başlatma/Sonlandırma İzleme

```python
import wmi
c = wmi.WMI()

# Yeni başlatılan işlemleri izle
watcher = c.Win32_Process.watch_for("creation")

while True:
    new_process = watcher()  # Bloklayıcı — yeni işlem başlayana kadar bekler
    print(f"Yeni işlem: {new_process.Name} (PID: {new_process.ProcessId})")
```

### 9.2. USB Cihaz Takma/Çıkarma İzleme

```python
import wmi
c = wmi.WMI()

# USB aygıt takma olayını izle
watcher = c.Win32_DeviceChangeEvent.watch_for(notification_type=2)  # 2 = Ekleme

print("USB cihaz bekleniyor...")
event = watcher()
print(f"Bir cihaz takıldı! Olay zamanı: {event.TIME_CREATED}")
```

## 10. Güvenlik Bilgileri

### 10.1. Antivirüs Durumu — `SecurityCenter2`

In [ ]:
import wmi

try:
    # SecurityCenter2 namespace'ine bağlanma
    c = wmi.WMI(namespace="root/SecurityCenter2")
    
    print("═" * 60)
    print("ANTİVİRÜS DURUMU")
    print("═" * 60)
    
    for av in c.AntiVirusProduct():
        print(f"\n  Ürün        : {av.displayName}")
        print(f"  Instance ID : {av.instanceGuid}")
        
    print("\n" + "═" * 60)
    print("GÜVENLİK DUVARI DURUMU")
    print("═" * 60)
    
    for fw in c.FirewallProduct():
        print(f"\n  Ürün : {fw.displayName}")
        
except Exception as e:
    print(f"SecurityCenter2 sorgusu başarısız: {e}")

## 11. Kullanıcı ve Hesap Yönetimi

### 11.1. Yerel Kullanıcılar — `Win32_UserAccount`

In [ ]:
import wmi
c = wmi.WMI()

print("═" * 60)
print("YEREL KULLANICI HESAPLARI")
print("═" * 60)

for user in c.Win32_UserAccount(LocalAccount=True):
    durum = "Aktif" if not user.Disabled else "Devre Dışı"
    print(f"\n  Kullanıcı   : {user.Name}")
    print(f"  Tam Ad      : {user.FullName}")
    print(f"  SID         : {user.SID}")
    print(f"  Durum       : {durum}")
    print(f"  Kilitli mi? : {'Evet' if user.Lockout else 'Hayır'}")

## 12. Başlangıç Programları

### 12.1. Windows Başlangıcında Çalışan Programlar — `Win32_StartupCommand`

In [ ]:
import wmi
c = wmi.WMI()

print("═" * 60)
print("BAŞLANGIÇ PROGRAMLARI")
print("═" * 60)

for startup in c.Win32_StartupCommand():
    print(f"\n  Program   : {startup.Name}")
    print(f"  Komut     : {startup.Command}")
    print(f"  Konum     : {startup.Location}")
    print(f"  Kullanıcı : {startup.User}")

## 13. Sık Kullanılan WMI Sınıfları Referans Tablosu

| Sınıf | Açıklama | Örnek Kullanım |
|-------|----------|----------------|
| `Win32_Processor` | İşlemci bilgileri | CPU model, hız, çekirdek sayısı |
| `Win32_PhysicalMemory` | RAM modül bilgileri | Kapasite, hız, DDR tipi |
| `Win32_VideoController` | Ekran kartı bilgileri | VRAM, sürücü versiyonu |
| `Win32_DiskDrive` | Fiziksel disk bilgileri | Model, kapasite, arayüz |
| `Win32_LogicalDisk` | Mantıksal disk bölümleri | C:, D:, boş alan |
| `Win32_BaseBoard` | Anakart bilgileri | Üretici, model |
| `Win32_BIOS` | BIOS bilgileri | Versiyon, tarih |
| `Win32_OperatingSystem` | İşletim sistemi | Windows versiyonu, RAM |
| `Win32_NetworkAdapterConfiguration` | Ağ ayarları | IP, MAC, DNS |
| `Win32_Service` | Windows servisleri | Durum, başlatma modu |
| `Win32_Process` | Çalışan işlemler | PID, bellek kullanımı |
| `Win32_PnPEntity` | Plug and Play aygıtlar | Tüm donanımlar |
| `Win32_PnPSignedDriver` | İmzalı sürücüler | Sürücü versiyonu, tarih |
| `Win32_UserAccount` | Kullanıcı hesapları | Kullanıcı adı, SID |
| `Win32_StartupCommand` | Başlangıç programları | Otomatik çalışan uygulamalar |
| `Win32_Battery` | Pil bilgileri (dizüstü) | Şarj durumu, kapasite |
| `Win32_Printer` | Yazıcılar | Ad, port, durum |
| `Win32_SoundDevice` | Ses cihazları | Ad, üretici |

## 14. Performans İpuçları ve En İyi Uygulamalar

1. **SELECT ile Sütun Belirleyin:** Tüm özellikleri çekmek yerine yalnızca ihtiyacınız olanları sorgulayın.
   ```python
   # Yavaş:
   c.Win32_Process()
   # Hızlı:
   c.query("SELECT Name, ProcessId FROM Win32_Process")
   ```

2. **`Win32_Product`'tan Kaçının:** Bu sınıf her çağrıldığında MSI paketlerini doğrular ve dakikalarca sürebilir. Alternatif: Registry sorgusu.

3. **WHERE ile Filtreleme:** Tüm sonuçları Python'da filtrelemek yerine WQL'de filtreleme yapın.
   ```python
   # Yavaş: Tüm servisleri çek, Python'da filtrele
   [s for s in c.Win32_Service() if s.State == "Running"]
   # Hızlı: WQL'de filtrele
   c.Win32_Service(State="Running")
   ```

4. **Bağlantıyı Tekrar Kullanın:** Her sorguda yeni `wmi.WMI()` nesnesi oluşturmak yerine tek bir bağlantıyı paylaşın.

5. **Hata Yönetimi:** WMI sorguları ağ veya izin sorunlarında `wmi.x_wmi` istisnası fırlatır. Mutlaka `try-except` bloğuyla sarın.